# 클러스터링 기반 추천 시스템(Clustering-Based Recommendation System)
- 유사한 사용자 그룹을 찾아 추천을 제공하는 방법
- 비지도 학습(unsupervised learning) 기법을 활용하여 사용자를 몇 개의 그룹(클러스터)으로 나누고, 같은 클러스터에 속한 사용자들이 선호하는 아이템을 추천하는 방식

- 동작 과정
1. 사용자 또는 아이템을 클러스터링
    - 사용자의 행동 데이터(구매 기록, 평점, 클릭 데이터 등)를 기반으로 유사한 사용자끼리 그룹화
    - 또는, 아이템을 클러스터링하여 유사한 상품을 찾는 방식도 가능
2. 클러스터별 대표적인 아이템 추천
    - 동일한 클러스터 내에서 인기 있는 아이템을 추천
    - 클러스터 중심(centroid)에 가까운 아이템 추천
    - 특정 사용자가 속한 클러스터의 다른 사용자의 선호 아이템을 추천
3. 새로운 사용자가 들어왔을 때 처리 방법
    - K-최근접 이웃(KNN) 기반으로 가장 유사한 클러스터를 찾아서 추천
    - 클러스터링을 주기적으로 업데이트하여 새로운 사용자의 변화를 반영

# 1.라이브러리 및 데이터 불러오기

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# 예시 데이터 (아이템의 콘텐츠 정보: 제목과 설명)
items = {
    '아이템': ['기생충', '부산행', '태극기 휘날리며', '도둑들', '설국열차', '범죄도시'],
    '설명': [
        '가난한 가족이 부잣집에 얽히며 벌어지는 이야기',
        '좀비 바이러스가 퍼진 세상에서 살아남기 위한 이야기',
        '한국 전쟁을 배경으로 형제의 이야기를 그린 영화',
        '도둑들이 모여 큰 한탕을 노리는 이야기',
        '미래의 기차 안에서 벌어지는 계급 투쟁 이야기',
        '형사가 조직 범죄와 싸우는 액션 영화'
    ]
}

In [3]:
ratings = {
    '사용자': ['사용자 A', '사용자 A', '사용자 A', '사용자 B', '사용자 B', '사용자 C'],
    '아이템': ['기생충', '부산행', '설국열차', '태극기 휘날리며', '도둑들', '범죄도시'],
    '평점': [5, 4, 3, 5, 3, 4]
}

In [4]:
# 데이터프레임으로 변환
items_df = pd.DataFrame(items)
ratings_df = pd.DataFrame(ratings)

In [5]:
items_df

,아이템,설명
0,기생충,가난한 가족이 부잣집에 얽히며 벌어지는 이야기
1,부산행,좀비 바이러스가 퍼진 세상에서 살아남기 위한 이야기
2,태극기 휘날리며,한국 전쟁을 배경으로 형제의 이야기를 그린 영화
3,도둑들,도둑들이 모여 큰 한탕을 노리는 이야기
4,설국열차,미래의 기차 안에서 벌어지는 계급 투쟁 이야기
5,범죄도시,형사가 조직 범죄와 싸우는 액션 영화


In [6]:
ratings_df

,사용자,아이템,평점
0,사용자 A,기생충,5
1,사용자 A,부산행,4
2,사용자 A,설국열차,3
3,사용자 B,태극기 휘날리며,5
4,사용자 B,도둑들,3
5,사용자 C,범죄도시,4


In [7]:
# TF-IDF 벡터라이저를 사용하여 설명을 벡터화
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(items_df['설명'])

In [8]:
# 사용자-아이템(영화) 평점 매트릭스 생성
user_item_matrix = ratings_df.pivot(index='사용자', columns='아이템', values='평점').fillna(0)
user_item_matrix

아이템,기생충,도둑들,범죄도시,부산행,설국열차,태극기 휘날리며
사용자,,,,,,
사용자 A,5.0,0.0,0.0,4.0,3.0,0.0
사용자 B,0.0,3.0,0.0,0.0,0.0,5.0
사용자 C,0.0,0.0,4.0,0.0,0.0,0.0


# 2.K-means 클러스터링

In [9]:
# k-means 클러스터링을 사용해서 사용자 클러스터링
num_clusters = 2
kmeans = KMeans(n_clusters=num_clusters, random_state=0)
user_clusters = kmeans.fit_predict(user_item_matrix)

In [10]:
user_item_matrix['클러스터'] = user_clusters

In [11]:
# 특정 클러스터의 다른 사용자들이 높게 평가한 아이템을 추천
def cluster_based_recommendations(user_id, num_recommendations):
    user_cluster = user_item_matrix.loc[user_id, '클러스터']    # 사용자가 속한 클러스터 찾기
    similar_users = user_item_matrix[user_item_matrix['클러스터'] == user_cluster].drop(user_id)  # 같은 클러스터에 속한 다른 사용자 찾기

    # 모든 사용자의 아이템 평점 평균 계산
    avg_ratings = similar_users.mean(axis=0).drop('클러스터')

    # 사용자가 이미 평가한 아이템 제외
    user_rated_items = user_item_matrix.loc[user_id].drop('클러스터')
    user_rated_items = user_rated_items[user_rated_items > 0].index
    avg_ratings = avg_ratings.drop(user_rated_items)

    # 상위 num_recommendations 개의 아이템 추천
    recommendations = avg_ratings.nlargest(num_recommendations).index.tolist()

    return recommendations

In [12]:
# 사용자 A에 대한 추천 계산(사용자 A에게 유사한 아이템 1개 추천)
recomm_items = cluster_based_recommendations('사용자 A', 1)

for item in recomm_items:
    print(item)

도둑들
